# Retail Sales Forecasting Example Notebook

This walkthrough demonstrates how to combine the project utilities to train and
evaluate an LSTM-based forecaster in JAX. It follows the storyboard described in
`retail_sales_forecasting_with_lstms.example.md`.

> Status (Midterm PR): data ingestion, feature engineering, and model training
> sections are scaffolded with TODO comments and placeholder cells.



In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from retail_sales_forecasting_utils import (
    DataSourceConfig,
    ModelConfig,
    TemporalFeatureConfig,
    build_feature_pipeline,
    ensure_data_root,
    evaluate_model,
    prepare_dataloader,
    train_model,
)

DATA_ROOT = Path("/data/store_sales")

data_cfg = DataSourceConfig(
    root_dir=DATA_ROOT,
    sales_file="train.parquet",
    calendar_file="holidays_events.csv",
    oil_file="oil.csv",
    transactions_file="transactions.csv",
    id_columns=("store_nbr", "family"),
    date_column="date",
    target_column="sales",
    frequency="D",
    horizon_days=28,
)

feature_cfg = TemporalFeatureConfig(
    include_external_regressors=True,
    include_promotions=True,
    lookback_days=90,
    train_ratio=0.75,
)
model_cfg = ModelConfig(hidden_size=96, epochs=8, batch_size=128)

ensure_data_root(data_cfg)

pipeline = list(build_feature_pipeline(data_cfg, feature_cfg))
train_ds, val_ds, metadata = prepare_dataloader(data_cfg, pipeline, feature_cfg)

print("Pipeline steps:", [fn.__name__ for fn in pipeline])
print("Train windows:", train_ds.inputs.shape, "| Validation windows:", val_ds.inputs.shape)
print("Feature columns:", len(metadata["feature_columns"]))



## 1. Data Snapshot

Inspect a few sequence windows to understand the per-store/per-family coverage.

In [ ]:
pd.concat([
    train_ds.ids.head(3).assign(split="train"),
    val_ds.ids.head(3).assign(split="validation"),
])

In [ ]:
sample_idx = 0
context_df = pd.DataFrame(
    train_ds.inputs[sample_idx],
    columns=metadata["feature_columns"],
)
context_df.head()

In [ ]:
pd.Series(train_ds.targets[sample_idx], name="scaled_target").head()

INFO  > cmd='/venv/lib/python3.12/site-packages/ipykernel_launcher.py -f /home/.local/share/jupyter/runtime/kernel-783e0930-1631-4d64-8bb4-f3a98bb74fcd.json'


## 2. Train the JAX LSTM

In [ ]:
trained_state = train_model(train_ds, val_ds, model_cfg, metadata)
trained_state["history"][:2]

## 3. Evaluate & Visualize

In [ ]:
artifacts = evaluate_model(trained_state, val_ds, metadata, metrics=("mae", "rmse", "mape"))
artifacts.metrics


### Plot forecasts for a validation series


In [ ]:
sample_meta = val_ds.ids.iloc[0]
store = sample_meta[metadata["id_columns"][0]]
family = sample_meta[metadata["id_columns"][1]]
series_df = artifacts.predictions[
    (artifacts.predictions[metadata["id_columns"][0]] == store)
    & (artifacts.predictions[metadata["id_columns"][1]] == family)
].copy()
series_df.sort_values("timestamp", inplace=True)

plt.figure(figsize=(10, 4))
plt.plot(series_df["timestamp"], series_df["target"], label="Actual", marker="o")
plt.plot(series_df["timestamp"], series_df["prediction"], label="Prediction", marker="x")
plt.title(f"Validation horizon — {store} / {family}")
plt.ylabel("Sales")
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()


### Next Steps

- Run hyperparameter sweeps (hidden size, learning rate) to tighten error bars.
- Extend the pipeline with external regressors such as oil price forecasts or
  macroeconomic indicators.
- Export predictions to Kaggle submission format for leaderboard evaluation.
